## Statistical tests

--> Sharpe (over time), Annual returns (over time)
1. Compare with existing methods performance (Equal weight, MVO, DFL, LLM)
2. Generate a number of random portfolios (then test for statistical significance above noise, i.e. null hypothesis)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from algorithm.compare_oracle_specificPortfolio import get_fpso, get_oracle, tickers
from algorithm.data_caller import get_data, normalize_wrds_data

tickers = ["AAPL", "AMZN", "JPM", "JNJ", "XOM", "WMT", "CAT", "NEE", "LIN", "AMT"]

raw_data = get_data(tickers=tickers)
normalized_data = normalize_wrds_data(raw_data)
print(raw_data.head())
print(normalized_data.head())
print(f"raw shape={raw_data.shape}, normalized shape={normalized_data.shape}")

fpso_returns, fpso_annualized, fpso_sharpe = get_fpso(
    tickers=tickers,
    epsilon=0.01,
    iterations=50,
    timesteps=100,
)
oracle_returns, oracle_annualized, oracle_sharpe = get_oracle(
    tickers=tickers,
    timesteps=100,
    transaction_cost=0.001,
)

print(f"FPSO final cumulative return: {fpso_returns[-1]:.6f}")
print(f"FPSO final annualized return: {fpso_annualized[-1]:.6f}")
print(f"Oracle final cumulative return: {oracle_returns[-1]:.6f}")
print(f"Oracle final annualized return: {oracle_annualized[-1]:.6f}")

wide = (
    raw_data.pivot_table(index="date", columns="permno", values="ret", aggfunc="last")
    .sort_index()
    .ffill()
    .fillna(0.0)
)

rng = np.random.default_rng(42)
random_final_returns = []
asset_cap = min(12, wide.shape[1])
for iteration in range(250):
    chosen = rng.choice(wide.columns.to_numpy(), size=asset_cap, replace=False)
    weights = rng.dirichlet(np.ones(len(chosen)))
    daily = wide[chosen].to_numpy(dtype=float) @ weights
    cumulative = np.cumprod(1.0 + daily)
    annualized = (1.0 + cumulative[-1]) ** (252.0 / len(daily)) - 1.0
    random_final_returns.append(annualized)

random_final_returns = np.asarray(random_final_returns, dtype=float)
observed = float(fpso_annualized[-1])
p_value = (np.sum(random_final_returns >= observed) + 1.0) / (len(random_final_returns) + 1.0)
print(f"Empirical p-value vs random baseline: {p_value:.6f}")

fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=False)

axes[0].plot(fpso_returns, label="FPSO cumulative", linewidth=2.0)
axes[0].plot(oracle_returns, label="Oracle cumulative", linewidth=2.0)
axes[0].set_title("Cumulative Performance")
axes[0].set_ylabel("Cumulative Return")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(fpso_annualized, label="FPSO annualized", linewidth=2.0)
axes[1].plot(oracle_annualized, label="Oracle annualized", linewidth=2.0)
axes[1].set_title("Annualized Performance")
axes[1].set_ylabel("Annualized Return")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].hist(random_final_returns, bins=30, alpha=0.75, color="slategray")
axes[2].axvline(observed, color="crimson", linewidth=2.0, label="FPSO observed")
axes[2].set_title("Random Baseline Distribution")
axes[2].set_xlabel("Final Annualized Return")
axes[2].set_ylabel("Count")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()
